In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import io
import typing

import mne
import numpy
import scipy
import pandas
import cairosvg
import PIL.Image
import matplotlib.pyplot as plt
import matplotlib.offsetbox as mob

import SDA.analytics

In [10]:
def get_silh(src_folder, subj, exp, result):
    best_result = pandas.read_csv(f"E:/CourseProject/{src_folder}/{subj}/{exp}/results/{result}/best_result.csv").iloc[0]
    return round(best_result['Avg-Silh'], 3)

def get_fmi(src_folder, subj, exp, result):
    best_result = pandas.read_csv(f"E:/CourseProject/{src_folder}/{subj}/{exp}/results/{result}/best_result.csv").iloc[0]
    return round(best_result['FMI'], 3)

In [11]:
def get_ci(data):
    data = numpy.array(data)
    alpha = 0.05
    n = len(data)
    std = data.std(ddof=1)
    t_crit = scipy.stats.t.ppf(1 - alpha / 2, df = n-1)
    return round(data.mean(), 3), round(t_crit * std / numpy.sqrt(n), 3)

def print_ci(stat, type):
    data = [ stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", type) for i in range(0, 153) ]
    print(f"{type} {stat.__name__[4:]}: {get_ci(data)[0]} +- {get_ci(data)[1]}")

print_ci(get_silh, "traditional")
print_ci(get_silh, "topological")
print_ci(get_silh, "combined")
print('-----------------------')
print_ci(get_fmi, "traditional")
print_ci(get_fmi, "topological")
print_ci(get_fmi, "combined")

traditional silh: 0.037 +- 0.008
topological silh: 0.191 +- 0.009
combined silh: 0.185 +- 0.009
-----------------------
traditional fmi: 0.834 +- 0.017
topological fmi: 0.892 +- 0.015
combined fmi: 0.889 +- 0.016


In [12]:
def print_sigma(stat, type_a, type_b):
    diffs = numpy.array([
        stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", type_b) -
        stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", type_a)
        for i in range(153)
    ], dtype=float)

    n = len(diffs)
    se = diffs.std(ddof=1) /numpy.sqrt(n)
    t = diffs.mean() / se
    p = 2 * scipy.stats.t.sf(abs(t), df=n - 1)

    print(f"{stat.__name__[4:]}: sigma = {abs(t):.6f}, p = {p:.3e}")


print_sigma(get_silh, "traditional", "topological")
print_sigma(get_fmi, "traditional", "topological")


silh: sigma = 25.750307, p = 2.651e-57
fmi: sigma = 12.663693, p = 1.506e-25


In [13]:
def get_ci(data, alpha = 0.05):
    data = numpy.array(data)
    n = len(data)
    std = data.std(ddof=1)
    t_crit = scipy.stats.t.ppf(1 - alpha / 2, df = n-1)
    return round(data.mean(), 3), round(t_crit * std / numpy.sqrt(n), 3)

def print_ci(stat, type):
    data = [ stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", type) for i in range(0, 153) ]
    print(f"{type} {stat.__name__[4:]}: {get_ci(data)[0]} +- {get_ci(data)[1]}; min = {numpy.percentile(data, 35):.4f}, max = {numpy.percentile(data, 50):.4f}")

print_ci(get_silh, "traditional")
print_ci(get_silh, "topological")
print_ci(get_silh, "combined")
print('-----------------------')
print_ci(get_fmi, "traditional")
print_ci(get_fmi, "topological")
print_ci(get_fmi, "combined")

traditional silh: 0.037 +- 0.008; min = 0.0270, max = 0.0300
topological silh: 0.191 +- 0.009; min = 0.1670, max = 0.1850
combined silh: 0.185 +- 0.009; min = 0.1620, max = 0.1770
-----------------------
traditional fmi: 0.834 +- 0.017; min = 0.8314, max = 0.8670
topological fmi: 0.892 +- 0.015; min = 0.8920, max = 0.9300
combined fmi: 0.889 +- 0.016; min = 0.8970, max = 0.9260


In [14]:
import scipy.stats

def test_diff_impl(topo, spec):
    diff = topo - spec
    mean_diff = numpy.mean(diff)
    sd_diff = numpy.std(diff, ddof = 1)
    se_diff = sd_diff / numpy.sqrt(len(diff))
    print(f"Topological: ", numpy.mean(topo))
    print(f"Spectral: ", numpy.mean(spec))
    print(f"Mean difference", mean_diff)
    print(f"Median difference", numpy.median(diff))
    print(f"SD of differences", sd_diff)
    print(f"SE of differences", se_diff)
    print(f"Proportion topo > spec: ", numpy.mean(diff > 0))

    t_crit = scipy.stats.t.ppf(1 - 0.05/2, len(diff) - 1)
    ci_low = mean_diff - t_crit * se_diff
    ci_high = mean_diff + t_crit * se_diff
    print(f"95% CI for mean difference: [{ci_low}, {ci_high}]")

    # Paired t-test; H1: mean(topo - spec) > 0
    t_stat, p_two_sided_t = scipy.stats.ttest_rel(topo, spec)
    if t_stat > 0:
        p_one_sided_t = p_two_sided_t / 2
    else:
        p_one_sided_t = 1 - p_two_sided_t / 2
    print("t statistic: ", t_stat)
    print("One-sided p-value", p_one_sided_t)

    # Wilcoxon signed-rank test; H1: topo > spec
    w_stat, p_wilcoxon = scipy.stats.wilcoxon(topo, spec, alternative='greater')
    print(f"Wilcoxon statistic: ", w_stat)
    print(f"One-sided p-value: ", p_wilcoxon)

    print(f"Cohen's dz: ", mean_diff / sd_diff)

    d_nz = diff[diff != 0]
    abs_ranks = scipy.stats.rankdata(numpy.abs(d_nz), method='average')
    r_plus, r_minus = numpy.sum(abs_ranks[d_nz > 0]), numpy.sum(abs_ranks[d_nz < 0])
    rank_biserial = (r_plus - r_minus) / (r_plus + r_minus)
    print(f"Rank-biserial correlation: ", rank_biserial)

def test_diff(stat, better, worse):
    better = [ stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", better) for i in range(0, 153) ]
    worse = [ stat("sleep-edf", f"Subj{i}", "exp_sleep_30_sec", worse) for i in range(0, 153) ]
    test_diff_impl(numpy.array(better), numpy.array(worse))

test_diff(get_silh, "topological", "traditional")
print('-----------------------')
test_diff(get_fmi, "topological", "traditional")

Topological:  0.19056209150326797
Spectral:  0.037111111111111116
Mean difference 0.15345098039215688
Median difference 0.154
SD of differences 0.07371111295710175
SE of differences 0.005959190284391473
Proportion topo > spec:  0.9934640522875817
95% CI for mean difference: [0.141677444332643, 0.16522451645167074]
t statistic:  25.750307184196025
One-sided p-value 1.3255918968385622e-57
Wilcoxon statistic:  11629.0
One-sided p-value:  7.112007409983592e-26
Cohen's dz:  2.0817889492655195
Rank-biserial correlation:  0.9741957389016213
-----------------------
Topological:  0.8919607843137255
Spectral:  0.8344967320261438
Mean difference 0.05746405228758169
Median difference 0.05599999999999994
SD of differences 0.05612826313849927
SE of differences 0.004537701127499897
Proportion topo > spec:  0.8496732026143791
95% CI for mean difference: [0.04849894372378086, 0.06642916085138252]
t statistic:  12.663692621651844
One-sided p-value 7.52926501222249e-26
Wilcoxon statistic:  10844.0
One-si